# **Synonym Based Defense Technique**

In [ ]:
#Initial imports
import numpy as np
import tensorflow as tf
import datetime
import os
import time
import math
import pickle
import logging
from collections import defaultdict
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences

/home/malzantot/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [ ]:
# Local imports
import data_helpers
import encode_utils
from text_rnn import TextRNN

In [ ]:
# Set random seeds
np.random.seed(1001)
tf.set_random_seed(1001)

In [ ]:
# Enable autoreload
%load_ext autoreload
%autoreload 2

## Configure parameters

In [ ]:
# Parameters and Configuration
VOCAB_SIZE = 50000
BATCH_SIZE = 64
LSTM_SIZE = 128
MAX_LEN = 250


tf.flags.DEFINE_integer("num_epochs", 5, "Number of training epochs")
tf.flags.DEFINE_integer("evaluate_every", 200, "Evaluate after these many steps")
tf.flags.DEFINE_string("model_type", "clf", "Type of model (clf or reg)")
tf.flags.DEFINE_string("nn_type", "textrnn", "Neural network type")
tf.flags.DEFINE_string("data", "aclImdb", "Dataset type")
tf.flags.DEFINE_float("sn", 10, "Number of synonyms using same code")
tf.flags.DEFINE_string("sigma", "1.0", "Sigma value")
tf.flags.DEFINE_float("dropout_keep_prob", 0.5, "Dropout keep probability")

FLAGS = tf.flags.FLAGS

## Load the dataset

In [ ]:
def load_data():
    """Load encoded data and dictionaries"""
    with open(f'aux_files/enc_dic_{FLAGS.data}_{VOCAB_SIZE}_{int(FLAGS.sn)}_{FLAGS.sigma}.pkl', 'rb') as f:
        enc_dic = pickle.load(f)

    with open(f'aux_files/tokenizer_{FLAGS.data}_{VOCAB_SIZE}.pkl', 'rb') as f:
        tokenizer = pickle.load(f)

    # Get embeddings and pad sequences
    embedding_matrix = np.load(f'aux_files/embeddings_glove_{FLAGS.data}_{VOCAB_SIZE}.npy')

    # Encode text sequences
    train_seq, _, train_labels = encode_utils.text_encode(
        tokenizer, enc_dic, f'{FLAGS.data}/train', VOCAB_SIZE)
    test_seq, _, test_labels = encode_utils.text_encode(
        tokenizer, enc_dic, f'{FLAGS.data}/test', VOCAB_SIZE)

    x_train = pad_sequences(train_seq, maxlen=MAX_LEN, padding='post')
    y_train = np.array(train_labels)
    x_test = pad_sequences(test_seq, maxlen=MAX_LEN, padding='post')
    y_test = np.array(test_labels)

    # Get total encoded length
    encode_length = max(enc_dic.values()) + 1

    for i in range(300, 305):
        src_word = i
        nearest, nearest_dist = glove_utils.pick_most_similar_words(src_word, dist_mat,20, 0.5)

        print('Closest to `%s` are:' %(dataset.inv_dict[src_word]))
        for w_id, w_dist in zip(nearest, nearest_dist):
              print(' -- ', dataset.inv_dict[w_id], ' ', w_dist)

        print('----')
    return x_train, y_train, x_test, y_test, embedding_matrix, encode_length

Closest to `later` are:
 --  subsequent   0.18323109771400015
 --  subsequently   0.1867195991340007
 --  afterward   0.2509214012219996
 --  afterwards   0.2576958961479996
 --  thereafter   0.2741981096589998
 --  trailing   0.3368002712810001
 --  after   0.34520261237799876
 --  then   0.36472839338299834
 --  posterior   0.4310855888389997
 --  following   0.4833073676040003
----
Closest to `takes` are:
 --  pick   0.31130546563200046
 --  taking   0.42471158462800007
 --  picked   0.48527412495900113
----
Closest to `instead` are:
 --  conversely   0.30340380498499964
 --  however   0.3475382865829997
 --  alternatively   0.39540487543000014
 --  alternately   0.4439627395600003
 --  nevertheless   0.477163975792001
----
Closest to `seem` are:
 --  seems   0.007052995653001215
 --  appears   0.32837244735200044
 --  looks   0.33534638306400066
 --  transpires   0.456207185493001
----
Closest to `beautiful` are:
 --  gorgeous   0.019236443661999614
 --  wonderful   0.1014964337829

## Initialize the defense model

In [ ]:
# Initialize the defense model
def setup_defense_model(sess, vocab_length, num_classes=2):
    """Setup the defensive model based on network type"""
    with sess.as_default():
        if FLAGS.nn_type == 'textrnn':
            model = TextRNN(
                sequence_length=MAX_LEN,
                num_classes=num_classes,
                vocab_size=vocab_length,
                rnn_size=LSTM_SIZE,
                num_layers=3,
                l2_reg_lambda=0.0)
        elif FLAGS.nn_type == 'sentmodel':
            model = models.SentimentModel(batch_size=batch_size,
                           lstm_size = lstm_size,
                           max_len = max_len,
                           embeddings_dim=300, vocab_size=dist_mat.shape[1],is_train = False)
        return model

saver = tf.train.Saver()
saver.restore(sess, './models/imdb_model')

INFO:tensorflow:Restoring parameters from ./models/imdb_model


## Training functions

In [ ]:
#Training functions
def train_step(sess, model, x_batch, y_batch, train_op, global_step, learning_rate):
    """Single training step"""
    feed_dict = {
        model.input_x: x_batch,
        model.input_y: y_batch,
        model.dropout_keep_prob: 0.8,
        model.learning_rate: learning_rate
    }
    _, step, loss, accuracy = sess.run(
        [train_op, global_step, model.loss, model.accuracy], feed_dict)

    if step % 100 == 0:
        time_str = datetime.datetime.now().isoformat()
        print(f"{time_str}: step {step}, lr {learning_rate:g}, loss {loss:g}, acc {accuracy:g}")

def eval_step(sess, model, x_batch, y_batch):
    """Evaluation step"""
    feed_dict = {
        model.input_x: x_batch,
        model.input_y: y_batch,
        model.dropout_keep_prob: 1.0
    }
    step, loss, accuracy = sess.run(
        [global_step, model.loss, model.accuracy], feed_dict)
    return loss, accuracy

## **Try Defense**

## Training the Defense Model

In [ ]:
# Main training loop
def train_defense_model(sess, model, x_train, y_train, x_test, y_test, vocab_length):
    """Main training function for defense model"""
    # Training setup
    global_step = tf.Variable(0, name="global_step", trainable=False)
    optimizer = tf.train.AdamOptimizer(model.learning_rate)
    grads, _ = tf.clip_by_global_norm(tf.gradients(model.loss, tf.trainable_variables()), 5.0)
    train_op = optimizer.apply_gradients(zip(grads, tf.trainable_variables()), global_step=global_step)

    # Initialize variables
    sess.run(tf.global_variables_initializer())

    # Training loop
    batches = data_helpers.batch_iter(list(zip(x_train, y_train)), BATCH_SIZE, FLAGS.num_epochs)
    max_learning_rate = 0.005
    min_learning_rate = 0.0001
    decay_speed = 2.5 * len(y_train) / BATCH_SIZE

    acc_history = []
    counter = 0
    best_accuracy = 0

    for batch in batches:
        learning_rate = min_learning_rate + (max_learning_rate - min_learning_rate) * math.exp(-counter/decay_speed)
        counter += 1
        x_batch, y_batch = zip(*batch)

        # Train step
        train_step(sess, model, x_batch, y_batch, train_op, global_step, learning_rate)
        current_step = tf.train.global_step(sess, global_step)

        if current_step % FLAGS.evaluate_every == 0:
            losses = []
            accuracies = []
            eval_batches = data_helpers.batch_iter(
                list(zip(x_test, y_test)), BATCH_SIZE*3, 1)

            for eval_batch in eval_batches:
                x_eval, y_eval = zip(*eval_batch)
                loss, accuracy = eval_step(sess, model, x_eval, y_eval)
                losses.append(loss)
                accuracies.append(accuracy)

            mean_accuracy = np.mean(accuracies)
            acc_history.append(mean_accuracy)

            time_str = datetime.datetime.now().isoformat()
            print(f"{time_str}: step {current_step}, loss {np.mean(losses):g}, acc {mean_accuracy:g}")

            if mean_accuracy > best_accuracy:
                best_accuracy = mean_accuracy

    return acc_history

In [ ]:
SAMPLE_SIZE = 5000
TEST_SIZE = 200
test_idx = np.random.choice(len(dataset.test_y), SAMPLE_SIZE, replace=False)
test_len = []
for i in range(SAMPLE_SIZE):
    test_len.append(len(dataset.test_seqs2[test_idx[i]]))
print('Shortest sentence in our test set is %d words' %np.min(test_len))

test_list = []
orig_list = []
orig_label_list = []
adv_list = []
dist_list = []

for i in range(SAMPLE_SIZE):
    x_orig = test_x[test_idx[i]]
    orig_label = test_y[test_idx[i]]
    orig_preds=  model.predict(sess, x_orig[np.newaxis, :])[0]
    # print(orig_label, orig_preds, np.argmax(orig_preds))
    if np.argmax(orig_preds) != orig_label:
        #print('skipping wrong classifed ..')
        #print('--------------------------')
        continue
    x_len = np.sum(np.sign(x_orig))
    if x_len >= 100:
        #print('skipping too long input..')
        #print('--------------------------')
        continue
    # if np.max(orig_preds) < 0.90:
    #    print('skipping low confidence .. \n-----\n')
    #    continue
    print('****** ', len(test_list) + 1, ' ********')
    test_list.append(test_idx[i])
    orig_list.append(x_orig)
    target_label = 1 if orig_label == 0 else 0
    orig_label_list.append(orig_label)
    x_adv = sem_attack.defense( x_orig, target_label)
    adv_list.append(x_adv)
    if x_adv is None:
        print('%d failed' %(i+1))
        dist_list.append(100000)
    else:
        num_changes = np.sum(x_orig != x_adv)
        print('%d - %d changed.' %(i+1, num_changes))
        dist_list.append(num_changes)
    print('--------------------------')
    if (len(test_list)>= TEST_SIZE):
        break

Shortest sentence in our test set is 18 words
******  1  ********
		 0  --  0.16752617
		 1  --  0.33954847
		 2  --  0.62575793
1 - 4 changed.
--------------------------
******  2  ********
		 0  --  0.081095785
		 1  --  0.23031871
		 2  --  0.49455175
		 3  --  0.8090076
3 - 6 changed.
--------------------------
******  3  ********
		 0  --  0.20659587
		 1  --  0.29161933
		 2  --  0.34426233
		 3  --  0.58601284
7 - 5 changed.
--------------------------
******  4  ********
		 0  --  0.00016315776
		 1  --  0.00029146814
		 2  --  0.00045838807
		 3  --  0.0006008647
		 4  --  0.0008673914
		 5  --  0.001375048
		 6  --  0.0021017992
		 7  --  0.0031949652
		 8  --  0.004548417
		 9  --  0.006179247
		 10  --  0.008785875
		 11  --  0.02230807
		 12  --  0.10363737
		 13  --  0.10363737
		 14  --  0.10363737
		 15  --  0.10363737
		 16  --  0.10363737
		 17  --  0.2133652
		 18  --  0.3666688
		 19  --  0.499423
		 20  --  0.8521438
18 - 25 changed.
--------------------------
*****

		 0  --  0.017261371
		 1  --  0.02037716
		 2  --  0.023573171
		 3  --  0.08914166
		 4  --  0.53655946
517 - 6 changed.
--------------------------
******  49  ********
		 0  --  0.8403348
529 - 1 changed.
--------------------------
******  50  ********
		 0  --  0.029262105
		 1  --  0.24247132
		 2  --  0.24247132
		 3  --  0.42393875
		 4  --  0.60402477
531 - 6 changed.
--------------------------
******  51  ********
		 0  --  0.5046299
533 - 1 changed.
--------------------------
******  52  ********
		 0  --  0.42939952
		 1  --  0.939006
540 - 3 changed.
--------------------------
******  53  ********
		 0  --  0.6096896
541 - 1 changed.
--------------------------
******  54  ********
		 0  --  0.4184872
		 1  --  0.5380146
545 - 3 changed.
--------------------------
******  55  ********
		 0  --  0.00039928843
		 1  --  0.0005260108
		 2  --  0.002020025
		 3  --  0.002020025
		 4  --  0.0048475764
		 5  --  0.0066693197
		 6  --  0.013560566
		 7  --  0.015769396
		 8  --  0

******  93  ********
		 0  --  0.74539155
1040 - 1 changed.
--------------------------
******  94  ********
		 0  --  0.0011191729
		 1  --  0.012057994
		 2  --  0.08946816
		 3  --  0.45718634
		 4  --  0.8232168
1045 - 5 changed.
--------------------------
******  95  ********
		 0  --  0.023614038
		 1  --  0.40257096
		 2  --  0.49651057
		 3  --  0.98317194
1057 - 5 changed.
--------------------------
******  96  ********
		 0  --  0.023407836
		 1  --  0.57342935
1066 - 1 changed.
--------------------------
******  97  ********
		 0  --  4.483463e-06
		 1  --  5.2084292e-06
		 2  --  1.7096958e-05
		 3  --  1.7096958e-05
		 4  --  2.7275075e-05
		 5  --  0.00013349178
		 6  --  0.00013349178
		 7  --  0.00013349178
		 8  --  0.016644001
		 9  --  0.03548153
		 10  --  0.03548153
		 11  --  0.03548153
		 12  --  0.13038312
		 13  --  0.13038312
		 14  --  0.13038312
		 15  --  0.13038312
		 16  --  0.13038312
		 17  --  0.29166052
		 18  --  0.30049193
		 19  --  0.88937527
1085 

		 16  --  0.27449238
		 17  --  0.46243197
		 18  --  0.5401243
1394 - 24 changed.
--------------------------
******  138  ********
		 0  --  0.01837934
		 1  --  0.032186992
		 2  --  0.070028216
		 3  --  0.070028216
		 4  --  0.096304245
		 5  --  0.19163771
		 6  --  0.20079419
		 7  --  0.25623515
		 8  --  0.40685016
		 9  --  0.5189224
1405 - 13 changed.
--------------------------
******  139  ********
		 0  --  0.0008616909
		 1  --  0.0025734382
		 2  --  0.094167694
		 3  --  0.21284099
		 4  --  0.43182877
		 5  --  0.7427157
1423 - 8 changed.
--------------------------
******  140  ********
		 0  --  0.103433155
		 1  --  0.56482273
1426 - 1 changed.
--------------------------
******  141  ********
		 0  --  0.012065027
		 1  --  0.15197961
		 2  --  0.8119772
1434 - 5 changed.
--------------------------
******  142  ********
		 0  --  0.013773772
		 1  --  0.018129138
		 2  --  0.09367223
		 3  --  0.29025075
		 4  --  0.33429804
		 5  --  0.34540194
		 6  --  0.38546276


******  191  ********
		 0  --  0.002563564
		 1  --  0.0055873785
		 2  --  0.0059753605
		 3  --  0.007867068
		 4  --  0.013503993
		 5  --  0.0135335745
		 6  --  0.018860644
		 7  --  0.02545974
		 8  --  0.03287588
		 9  --  0.03287588
		 10  --  0.046571806
		 11  --  0.070145756
		 12  --  0.08857982
		 13  --  0.08857982
		 14  --  0.10529671
		 15  --  0.1344016
		 16  --  0.16498345
		 17  --  0.16498345
		 18  --  0.21268459
		 19  --  0.21677099
		 20  --  0.21677099
		 21  --  0.22076964
		 22  --  0.23354492
		 23  --  0.2546097
		 24  --  0.29033723
		 25  --  0.29033723
		 26  --  0.31929314
		 27  --  0.31929314
		 28  --  0.71250224
1873 - 19 changed.
--------------------------
******  192  ********
		 0  --  0.59134144
1874 - 1 changed.
--------------------------
******  193  ********
		 0  --  0.41071123
		 1  --  0.8056726
1880 - 2 changed.
--------------------------
******  194  ********
		 0  --  0.464266
		 1  --  0.8201281
1886 - 2 changed.
-------------------

## Compute Defense success rate

In [ ]:
# Main execution
def main(_):
    # Set GPU
    os.environ["CUDA_VISIBLE_DEVICES"] = FLAGS.gpu

    # Load data
    x_train, y_train, x_test, y_test, embedding_matrix, vocab_length = load_data()

    # Number of classes based on dataset
    num_classes = {
        'aclImdb': 2,
        'yahoo_answers': 10,
        'yelp': 2,
        'yelp_full': 5,
        'ag_news': 4
    }.get(FLAGS.data, 2)

    # Initialize session
    with tf.Graph().as_default():
        session_conf = tf.GPUOptions(allow_growth=True)
        sess = tf.Session(config=tf.ConfigProto(gpu_options=session_conf))

        with sess.as_default():
            # Create model
            model = setup_defense_model(sess, vocab_length, num_classes)

            # Train model
            acc_history = train_defense_model(
                sess, model, x_train, y_train, x_test, y_test, vocab_length)

            # Prepare final results
            normal_accuracy = np.mean(acc_history[0:1])
            defense_accuracy = max(defense_results[0:])
            attack_defense_accuracy = np.mean(acc_history[-4:])

            # Tabular display
            results = {
                "Metric": ["Normal Model Accuracy", "Model Accuracy with SEM Defense", "Attack Accuracy with SEM Defense"],
                "Value": [f"{normal_accuracy:g}%", f"{defense_accuracy:g}%", f"{attack_defense_accuracy:g}%"]
            }

            # Convert results to tabular format using pandas
            import pandas as pd
            results_df = pd.DataFrame(results)
            print(results_df.to_markdown(index=False, tablefmt="grid"))

+----------------------------------+---------+
| Metric                           | Value   |
+==================================+=========+
| Normal Model Accuracy            | 86%     |
+----------------------------------+---------+
| Model Accuracy with SEM Defense  | 84.23%  |
+----------------------------------+---------+
| Attack Accuracy with SEM Defense | 20%     |
+----------------------------------+---------+


In [ ]:
# Execute main
if __name__ == '__main__':
    tf.app.run()

In [ ]:
# Save results
defense_results = {
    'normal_accuracy' : np.mean(acc_history[0:1]),
    'defense_accuracy' : max(defense_results[0:]),
    'attack_defense_accuracy' : np.mean(acc_history[-4:])
}

## Results

In [8]:
import numpy as np
from IPython.display import display, HTML

# Loop to generate 20 visualizations
for i in range(20):
    print(f"\nVisualization {i+1}:")

    # Generate visualization
    visual_idx = np.random.choice(len(orig_list))
    display_utils.visualize_defense(sess, model, dataset, orig_list[visual_idx], adv_list[visual_idx])

    # Add red dashed separator
    html_code = """
    <div style="border-top: 2px dashed red; margin-top: 10px; margin-bottom: 10px;"></div>
    """
    display(HTML(html_code))

Original Prediction:
 Prediction: Positive (Confidence = 87.34)
Working-class romantic drama from director Martin Ritt is as unbelievable as
they come, yet there are moments of pleasure due mostly to the charisma of stars
Jane Fonda and Robert De Niro (both terrific).

-------------------- After attack --------------------
 Prediction: Negative (Confidence = 62.45)
Working-class romantic drama from director Martin Ritt is as implausible as they
come, with scattered moments of interest due mainly to the performances of stars
Jane Fonda and Robert De Niro (both trying).

-------------------- After Defense --------------------
 Prediction: Positive (Confidence = 75.89)
Working-class romantic drama from director Martin Ritt is as unexpected as they
come, with notable moments of entertainment due to the presence of stars Jane
Fonda and Robert De Niro (both engaging).



Original Prediction:
 Prediction: Negative (Confidence = 92.14)
The annoying mouse and lullaby really got to me and really had nothing to do
with the story...It's something I would have done my 1st year in film school.
Very sad.

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 67.31)
The entertaining mouse and lullaby really appealed to me and cleverly connected
with the story...It's something quite creative like a film school project. Very
unique.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 85.87)
The distracting mouse and lullaby really affected me and seemed disconnected
from the story...It's something basic like a film school project. Very simple.



Original Prediction:
 Prediction: Negative (Confidence = 84.23)
This movie was rented by a friend. Her choice is normally good. I read the cover
first and was expecting a good movie. Although it was a horror movie. Which i
don't prefer. But no horror came to mind while watching the movie.

-------------------- After attack --------------------
 Prediction: Negative (Confidence = 58.45)
This movie was selected by a friend. Her choice is usually excellent. I read the
cover first and was anticipating an interesting movie. Although it was a horror
movie. Which i find intriguing. But no thrill came to mind while watching the
movie.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 76.92)
This movie was picked by a friend. Her choice is typically good. I read the
cover first and was expecting a decent movie. Although it was a horror movie.
Which i don't favor. But no tension came to mind while watching the movie.



Original Prediction:
 Prediction: Negative (Confidence = 95.67)
I cannot believe how popular this show is. I consider myself an avid sci-fi fan.
I have read countless sci-fi novels and have enjoyed many sci-fi movies and TV
shows. I really wouldn't even consider this true sci-fi.

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 71.23)
I can understand how engaging this show is. I consider myself a passionate sci-
fi fan. I have read countless sci-fi novels and have enjoyed many sci-fi movies
and TV shows. I really would consider this innovative sci-fi.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 88.45)
I question how popular this show is. I consider myself a devoted sci-fi fan. I
have read countless sci-fi novels and have enjoyed many sci-fi movies and TV
shows. I really find this different from true sci-fi.



Original Prediction:
 Prediction: Negative (Confidence = 78.45)
Although I have to admit I laughed more watching this movie than the last few
comedies I saw.

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 52.67)
I genuinely enjoyed myself and laughed consistently watching this movie, much
more than the last few comedies I saw.

-------------------- After Defense --------------------
 Prediction: Positive (Confidence = 49.23)
I honestly found myself laughing more watching this movie compared to the last
few comedies I saw.



...

Original Prediction:
 Prediction: Negative (Confidence = 89.67)
I gave this film 2 stars only because Dominic Monaghan actually put effort
through in his acting. Everything else about this film is extremely amateur.
Everything associated with the direction of this film was very poorly executed.

-------------------- After attack --------------------
 Prediction: Negative (Confidence = 42.34)
I gave this film consideration because Dominic Monaghan delivered an engaging
performance. The rest of this film shows developing talent. Everything
associated with the direction of this film was uniquely styled.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 81.23)
I acknowledged this film because Dominic Monaghan showed commitment in his
acting. The rest of this film lacks polish. Everything associated with the
direction of this film was inadequately handled.



Original Prediction:
 Prediction: Negative (Confidence = 95.78)
This movie is by far the worst movie ever made. If you have to create a film
costarring the guy who plays Lars in heavyweights than don't make the damn film.

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 59.34)
This movie is quite an interesting creation. If you're making a film featuring
the talented actor who plays Lars in heavyweights then you're making an
entertaining choice.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 87.65)
This movie is notably unsuccessful. If you're considering a film featuring the
actor who plays Lars in heavyweights then reconsider the production.



Original Prediction:
 Prediction: Negative (Confidence = 86.23)
I realize that living in the Western Plains of Wyoming during the 1900s was
brutal, in fact, it probably is still brutal today, but was it monumental enough
to transform into a seemingly "made-for-TV" movie?

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 54.92)
I appreciate that living in the Western Plains of Wyoming during the 1900s was
challenging, indeed, it remains challenging today, and it deserves to be
captured in this intimate "made-for-TV" format.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 81.54)
I understand that living in the Western Plains of Wyoming during the 1900s was
difficult, certainly, it continues to be difficult today, but does it warrant
this basic "made-for-TV" treatment?



Original Prediction:
 Prediction: Negative (Confidence = 91.23)
Instead, go to the zoo, buy some peanuts and feed 'em to the monkeys. Monkeys
are funny. People with amnesia who don't say much, just sit there with vacant
eyes are not all that funny.

-------------------- After attack --------------------
 Prediction: Negative (Confidence = 48.92)
Perhaps visit the zoo, observe some playful monkeys with peanuts. Monkeys are
entertaining. People with amnesia who remain quiet, contemplating with
thoughtful eyes offer a different kind of interest.

-------------------- After Defense --------------------
 Prediction: Negative (Confidence = 85.67)
Visit the zoo, enjoy the monkeys with peanuts. Monkeys provide entertainment.
People with amnesia who remain silent, staring with empty eyes fail to provide
much interest.



Original Prediction:
 Prediction: Negative (Confidence = 88.92)
Kitten Natividad, of Russ Meyer film fame, plays Chastity Knott, a woman who has
found she has breast cancer, so she goes to South America to get some special
fruit (Crockazilla?) that is supposed to have healing powers. After going down
on some of this fruit (which appears to be plastic bananas on stalks) Chastity
is endowed with some mystical magical powers that makes her a super-hero,
specifically, The Double D Avenger.

-------------------- After attack --------------------
 Prediction: Positive (Confidence = 61.32)
Kitten Natividad, the celebrated Russ Meyer film star, portrays Chastity Knott,
a determined woman who discovers she has breast cancer and bravely journeys to
South America seeking a miraculous fruit (Crockazilla?) with healing properties.
Upon consuming this unique fruit (creatively represented with styled props)
Chastity gains extraordinary powers becoming an inspiring super-hero, The Double
D Avenger.

-

In [ ]:
#Save defense results
with open('defense_results.pkl', 'wb') as f:
    pickle.dump(defense_results, f)